# YOLO OBB training (Phase A + B) — Google Colab

## Hardware: use a **GPU** runtime, not TPU

This project trains with **Ultralytics YOLO** on **PyTorch**. Ultralytics officially supports **CUDA GPUs** and **Apple MPS**, not Google Cloud TPUs.

Colab **TPUs** run workloads through **XLA** (JAX or PyTorch XLA). The Ultralytics trainer is built around standard PyTorch CUDA/CPU paths and **does not implement a supported TPU training path**. Trying to force TPU will generally fail or require a full rewrite of the training loop.

**What to do instead:** in Colab go to **Runtime → Change runtime type → Hardware accelerator → GPU** (e.g. T4, L4, A100). The cells below use `device=0` when CUDA is available.

## Dataset

The GitHub repo does **not** include `data/augmented/` (see `README.md`). **Clone** the repo (or copy it to Drive), then either **extract your shared augmented zip** into `data/augmented/` at the project root, or run `colab_augment_dataset.ipynb` / `augment.py` on raw data. You need `dataset.yaml`, `configs/`, and a populated `data/augmented/` before training. Set `PROJECT_ROOT` in the next section.

## 1) Optional: mount Google Drive

Skip if you use `git clone` into `/content` only.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; skip Drive mount.")

## 2) Project path and dependencies

Set `PROJECT_ROOT` to the folder that contains `dataset.yaml`, `configs/`, and `data/augmented/`.

In [ ]:
import os
from pathlib import Path

# --- EDIT THIS ---
PROJECT_ROOT = Path("/content/drive/MyDrive/ups_yolo")  # or Path("/content/ups_yolo")

os.chdir(PROJECT_ROOT)
assert (PROJECT_ROOT / "dataset.yaml").is_file(), f"Missing dataset.yaml under {PROJECT_ROOT}"
assert (PROJECT_ROOT / "configs" / "yolo26s_finetune.yaml").is_file(), "Missing configs/yolo26s_finetune.yaml"
print("cwd:", Path.cwd())

In [ ]:
!pip install -q ultralytics albumentations opencv-python-headless "numpy>=1.26,<2" pyyaml rich

## 3) Point `dataset.yaml` at this machine

`augment.py` writes a Windows-style absolute `path`. On Colab we rewrite `path` to the augmented dataset root so Ultralytics can find images.

In [ ]:
import yaml

ds_path = PROJECT_ROOT / "dataset.yaml"
data = yaml.safe_load(ds_path.read_text(encoding="utf-8"))
data["path"] = str((PROJECT_ROOT / "data" / "augmented").resolve())
ds_path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
print("Updated dataset path:", data["path"])

## 4) Device check (GPU vs CPU)

In [ ]:
import torch

if torch.cuda.is_available():
    train_device = 0
    print("Using CUDA:", torch.cuda.get_device_name(0))
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    train_device = "mps"
    print("Using MPS")
else:
    train_device = "cpu"
    print(
        "WARNING: No GPU detected. In Colab: Runtime → Change runtime type → GPU. "
        "TPU runtime will NOT accelerate this Ultralytics training."
    )

## 5) Phase A — frozen backbone (20 epochs)

Same hyperparameters as `scripts/train_phase_a.py` (no interactive resume prompt).

In [ ]:
from ultralytics import YOLO

PHASE_A_DIR = PROJECT_ROOT / "runs" / "phase_a"
PHASE_A_DIR.mkdir(parents=True, exist_ok=True)

model_a = YOLO("yolo26s-obb.pt")
model_a.train(
    data=str(ds_path.resolve()),
    epochs=20,
    imgsz=640,
    batch=16,
    freeze=10,
    task="obb",
    cfg=str(PROJECT_ROOT / "configs" / "yolo26s_finetune.yaml"),
    project=str(PROJECT_ROOT / "runs"),
    name="phase_a",
    exist_ok=True,
    val=True,
    plots=True,
    save=True,
    device=train_device,
    save_dir=str(PHASE_A_DIR),
)

## 6) Phase B — full fine-tune (100 epochs)

Loads `runs/phase_a/weights/best.pt` — same settings as `scripts/train_phase_b.py`.

In [ ]:
phase_a_weights = PROJECT_ROOT / "runs" / "phase_a" / "weights" / "best.pt"
assert phase_a_weights.is_file(), f"Missing {phase_a_weights}; run Phase A first."

PHASE_B_DIR = PROJECT_ROOT / "runs" / "phase_b"
PHASE_B_DIR.mkdir(parents=True, exist_ok=True)

model_b = YOLO(str(phase_a_weights))
model_b.train(
    data=str(ds_path.resolve()),
    epochs=100,
    imgsz=640,
    batch=8,
    freeze=0,
    task="obb",
    cfg=str(PROJECT_ROOT / "configs" / "yolo26s_finetune.yaml"),
    lr0=0.0001,
    lrf=0.01,
    cos_lr=True,
    project=str(PROJECT_ROOT / "runs"),
    name="phase_b",
    exist_ok=True,
    val=True,
    plots=True,
    save=True,
    patience=15,
    device=train_device,
    save_dir=str(PHASE_B_DIR),
)

## 7) Download weights (optional)

Best Phase B checkpoint: `runs/phase_b/weights/best.pt` under `PROJECT_ROOT`.

In [ ]:
from google.colab import files

best_b = PROJECT_ROOT / "runs" / "phase_b" / "weights" / "best.pt"
if best_b.is_file():
    files.download(str(best_b))
else:
    print("best.pt not found:", best_b)